# Four-dimensional Neuroimaging Data

The neuroimpy package contains data structures and functions for reading, accessing, and processing 4-dimensional neuroimaging data.

## Reading a four-dimensional NIFTI image

To read a 4D image, use the `read_vec()` function

In [ ]:
# Import required libraries
import neuroimpy as pn
import numpy as np

# Create example 4D data for tutorial
# (In practice, you would use real neuroimaging files)

# Create a 4D space (64x64x25 spatial, 100 time points)
space_4d = pn.NeuroSpace(dim=(64, 64, 25, 100), 
                         spacing=(3.5, 3.5, 5.0, 2.0),
                         origin=(-110.5, -88.9342, -42.75, 0.0))

# Create example 4D data with some temporal structure
np.random.seed(42)
example_4d_data = np.random.randn(64, 64, 25, 100)

# Create a 4D NeuroVec
vec = pn.DenseNeuroVec(example_4d_data, space_4d)

# Note: In practice, you would read a 4D NIFTI file like this:
# vec = pn.read_vec("path/to/4d_image.nii")

print(vec.shape)
print(vec)

## Reading multiple 4D images

In [ ]:
# Create multiple example 4D images
file_list = ["image1.nii", "image2.nii", "image3.nii"]

# In practice, you would read multiple files:
# vec = pn.read_vec(file_list)

# For this tutorial, let's create and concatenate multiple 4D datasets
vec1 = pn.DenseNeuroVec(np.random.randn(64, 64, 25, 50), 
                        pn.NeuroSpace(dim=(64, 64, 25, 50)))
vec2 = pn.DenseNeuroVec(np.random.randn(64, 64, 25, 50), 
                        pn.NeuroSpace(dim=(64, 64, 25, 50)))

# Concatenate along time dimension
vec_concat = vec1.concat(vec2)
print(f"Concatenated shape: {vec_concat.shape}")
# Will concatenate along the time dimension

## Extracting subsets of volumes

In [ ]:
# Extract subset of volumes
vec_subset = vec.sub_vector(slice(0, 6))
print(f"Subset shape (first 6 volumes): {vec_subset.shape}")

# Extract specific volumes by indices
vec_subset = vec.sub_vector([0, 2, 4, 6])
print(f"Subset shape (specific volumes): {vec_subset.shape}")

## Extracting time series data

In [ ]:
# Extract time series from a single voxel
ts = vec.series(10, 10, 10)
print(f"Time series shape: {ts.shape}")

# Plot the time series
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 4))
plt.plot(ts)
plt.xlabel('Time point')
plt.ylabel('Signal')
plt.title('Time series at voxel (10, 10, 10)')
plt.show()

## Multiple voxel time series

In [ ]:
# Extract time series from multiple voxels
coords = np.array([[10, 10, 10],
                   [20, 20, 10],
                   [30, 30, 10]])
                   
ts_matrix = vec.series(coords)
print(f"Time series matrix shape: {ts_matrix.shape}")
print(f"(time points x voxels): ({ts_matrix.shape[0]} x {ts_matrix.shape[1]})")

## ROI-based time series extraction

In [ ]:
# Create a spherical ROI
center = [32, 32, 12]
roi = pn.spherical_roi(vec.space, center, radius=8)

# Extract time series for all voxels in the ROI
roi_series = vec.series_roi(roi)
print(f"ROI time series shape: {roi_series.shape}")
print(f"Number of voxels in ROI: {roi_series.shape[1]}")

## Using linear indices

In [ ]:
# Extract time series using linear indices
# First 100 voxels
roi_series = vec.series(np.arange(100))
print(f"Time series shape for first 100 voxels: {roi_series.shape}")

## Working with masks

In [ ]:
# Create a binary mask for tutorial
# In practice: mask = pn.read_vol("brain_mask.nii")

# Create example mask (sphere in center of brain)
mask_data = np.zeros((64, 64, 25))
center = np.array([32, 32, 12])
for i in range(64):
    for j in range(64):
        for k in range(25):
            if np.sqrt((i-center[0])**2 + (j-center[1])**2 + (k-center[2])**2) <= 15:
                mask_data[i, j, k] = 1

mask = pn.LogicalNeuroVol(mask_data, vec.space.sub_space())

# Get indices of non-zero voxels
mask_indices = np.where(mask.data.ravel(order='F'))[0]

# Extract time series for masked voxels
masked_series = vec.series(mask_indices)
print(f"Extracted {masked_series.shape[1]} voxels from mask")

## Converting between formats

In [ ]:
# Convert to sparse representation using a mask
mask_vol = pn.LogicalNeuroVol(mask.data > 0, mask.space)
sparse_vec = vec.as_sparse(mask_vol)
print(f"Sparse vector type: {type(sparse_vec)}")

# Sparse vectors work the same way
ts = sparse_vec.series(10, 10, 10)
print(f"Time series from sparse vector: shape={ts.shape}")

## Converting to matrix format

In [ ]:
# Get as voxels x time matrix
matrix = vec.as_matrix()
print(f"Matrix shape: {matrix.shape}")
print(f"(voxels x time): ({matrix.shape[0]} x {matrix.shape[1]})")

## Concatenating NeuroVecs

In [ ]:
# Create some example vectors with matching spatial dimensions
space1 = pn.NeuroSpace(dim=(10, 10, 10, 50))
space2 = pn.NeuroSpace(dim=(10, 10, 10, 30))

vec1 = pn.DenseNeuroVec(np.random.randn(10, 10, 10, 50), space1)
vec2 = pn.DenseNeuroVec(np.random.randn(10, 10, 10, 30), space2)

# Concatenate
combined = vec1.concat(vec2)
print(f"Combined shape: {combined.shape}")
print(f"Original shapes: {vec1.shape} + {vec2.shape} = {combined.shape}")

## Extracting individual volumes

In [ ]:
# Get the first volume
vol0 = vec[..., 0]
print(f"First volume type: {type(vol0)}")
print(f"First volume shape: {vol0.shape}")

# Get multiple volumes as a list
vols = vec.vols([0, 10, 20, 30])
print(f"Number of volumes extracted: {len(vols)}")
print(f"Each volume shape: {vols[0].shape}")

## Time series preprocessing

In [ ]:
# Center and scale each time series
vec_scaled = vec.scale_series(center=True, scale=True)

# Verify: each voxel should have mean≈0, std≈1
ts = vec_scaled.series(10, 10, 10)
print(f"Mean: {np.mean(ts):.6f}, Std: {np.std(ts):.6f}")

# Check a few more voxels
for coord in [(20, 20, 10), (30, 30, 15)]:
    ts = vec_scaled.series(*coord)
    print(f"Voxel {coord}: Mean={np.mean(ts):.6f}, Std={np.std(ts):.6f}")

## Memory-mapped vectors

In [ ]:
# Create a memory-mapped vector for large datasets
# This is useful when data is too large to fit in memory

# Create example large dataset dimensions
print("Creating memory-mapped vector for large dataset...")
space_big = pn.NeuroSpace(dim=(100, 100, 50, 1000))

# For tutorial, we'll create a smaller example
# In practice, you'd use this for truly large datasets
space_demo = pn.NeuroSpace(dim=(50, 50, 25, 200))
big_data = np.random.randn(50, 50, 25, 200)

# Create BigNeuroVec - keeps data on disk
big_vec = pn.BigNeuroVec(big_data, space_demo)

# Access works the same way
ts = big_vec.series(25, 25, 12)
print(f"Time series shape from BigNeuroVec: {ts.shape}")

# Data stays on disk until accessed
print(f"BigNeuroVec shape: {big_vec.shape}")
print(f"BigNeuroVec type: {type(big_vec)}")